# 🌍 Geologist Expert Training - MINIMAL VERSION
Uses TinyLlama (fits easily on T4) with basic Trainer.

1. Upload `landsat_expert_training_data.jsonl`
2. Select T4 GPU in Runtime settings
3. Run all!

In [ ]:
# Step 1: Install (NO version pins - let Colab resolve)
!pip install -q transformers peft accelerate bitsandbytes datasets

In [ ]:
# Step 2: Imports (NO trl - avoid diffusers conflicts)
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import load_dataset

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
# Step 3: Use TinyLlama (small, OPEN, fast)
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
MAX_SEQ_LENGTH = 1024

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

In [ ]:
# Step 4: Load Model
print("📦 Loading TinyLlama...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
print("✅ Model loaded!")

In [ ]:
# Step 5: LoRA Config
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [ ]:
# Step 6: Load & Format Data
dataset = load_dataset("json", data_files="landsat_expert_training_data.jsonl", split="train")
print(f"📂 Loaded {len(dataset)} examples")

def tokenize(example):
    text = f"### Question:\n{example['instruction']}\n\n### Answer:\n{example['output']}"
    tokens = tokenizer(text, truncation=True, max_length=MAX_SEQ_LENGTH, padding="max_length")
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
print("✅ Data tokenized!")

In [ ]:
# Step 7: Train!
training_args = TrainingArguments(
    output_dir="./outputs",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=100,
    warmup_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

print("🏋️ Training... (~5 min)")
trainer.train()
print("✅ Done!")

In [ ]:
# Step 8: Save & Download
model.save_pretrained("geologist-lora")
tokenizer.save_pretrained("geologist-lora")
!zip -r geologist-lora.zip geologist-lora/
print("\n✅ Download 'geologist-lora.zip' from Files panel!")